# Task 3 - ML Model Portfolio (Week 3)
**Notebook:** `notebooks/Task3.ipynb`
**Aligned lectures:** ML Algorithms on Spark; Model Training & Tuning at Scale; Unsupervised Learning I.

Four classifiers tuned with `CrossValidator`, with training time and key metrics in a summary table.

**Algorithm choice (justified for high-dimensional sparse text):**
1. **Logistic Regression** - the strong, interpretable linear baseline for TF-IDF; weights map to words.
2. **Naive Bayes (multinomial)** - the classic probabilistic text classifier; fast, surprisingly strong on bag-of-words.
3. **Linear SVM (LinearSVC)** - maximum-margin linear model, robust in high dimensions.
4. **Random Forest** - non-linear ensemble baseline; included to show how tree models cope with sparse text features (a useful comparison/limitation).

In [1]:
# === Colab bootstrap: install Spark (run once per session) ===
# Uninstall conflicting package 'dataproc-spark-connect' if present, which expects pyspark v4+
!pip uninstall -y dataproc-spark-connect
!pip install pyspark==3.5.1
import os, time
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("7006SCN_AmazonReviews_Sentiment")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "12g")
         .config("spark.driver.maxResultSize", "2g")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Found existing installation: dataproc-spark-connect 1.1.0
Uninstalling dataproc-spark-connect-1.1.0:
  Successfully uninstalled dataproc-spark-connect-1.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.5 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=d3ea1db083bab7b1e9b1325fa33c2d4b7fe401264a238cec7d0bace37b2cc028
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.3
    Uninstalling pyspark-4.0.3:
      Successfully uninstalled pyspark-4.0.3
Spark version: 3.5.1
Spark UI: http:/

In [2]:
# Mount Google Drive so data/models persist across the six notebooks.
from google.colab import drive
drive.mount('/content/drive')
BASE   = '/content/drive/MyDrive/7006SCN'
RAW    = BASE + '/raw'                      # downloaded jsonl.gz
PARQ   = BASE + '/reviews_parquet'          # raw -> parquet
PROC   = BASE + '/reviews_processed'        # after Task2 NLP pipeline
EXPORT = BASE + '/tableau'                  # Task6 exports
import os
for p in (BASE, RAW, EXPORT):
    os.makedirs(p, exist_ok=True)
print('Base:', BASE)

Mounted at /content/drive
Base: /content/drive/MyDrive/7006SCN


In [3]:
# Optional: sample for tractable CV on free Colab. 1.0 = full data.
TRAIN_FRAC = 0.005 # Further reduced to a smaller fraction for manageability
train = spark.read.parquet(PROC + "_train")
if TRAIN_FRAC < 1.0:
    train = train.sample(False, TRAIN_FRAC, seed=42)
# Added sampling for the test DataFrame as well to prevent resource exhaustion during evaluation
test = spark.read.parquet(PROC + "_test")
if TRAIN_FRAC < 1.0: # Apply sampling to test set with the same fraction
    test = test.sample(False, TRAIN_FRAC, seed=42)
train = train.cache(); test = test.cache()
print("train", train.count(), "test", test.count())

train 75423 test 18758


In [4]:
from pyspark.ml.classification import (LogisticRegression, NaiveBayes,
                                       LinearSVC, RandomForestClassifier)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time

auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
f1_eval  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
acc_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

# Rewritten tune function to explicitly accept the training DataFrame
# This allows the user to pass a sampled DataFrame if needed.
def tune(name, est, grid, train_df_for_tuning, folds=3):
    # Ensure the passed training DataFrame is cached if it's large and used repeatedly
    train_df_for_tuning.cache().count()

    cv = CrossValidator(estimator=est, estimatorParamMaps=grid, evaluator=auc_eval,
                        numFolds=folds, parallelism=2, seed=42)
    t0=time.time(); m=cv.fit(train_df_for_tuning); secs=time.time()-t0 # Use the provided DataFrame
    best=m.bestModel; pred=best.transform(test)
    row={"model":name,"train_secs":round(secs,1),
         "AUC":round(auc_eval.evaluate(pred),4),
         "F1":round(f1_eval.evaluate(pred),4),
         "Accuracy":round(acc_eval.evaluate(pred),4),
         "best_params":{p.name:v for p,v in best.extractParamMap().items()
                        if p.name in ("regParam","elasticNetParam","smoothing","maxIter","numTrees","maxDepth")}}
    print(name, row); return best,row,pred

## 3.1 Logistic Regression

In [5]:
lr=LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
lr_grid=(ParamGridBuilder().addGrid(lr.regParam,[0.0,0.1])
         .addGrid(lr.elasticNetParam,[0.0,0.1]).build())
lr_best,lr_row,lr_pred=tune("LogisticRegression",lr,lr_grid, train)

LogisticRegression {'model': 'LogisticRegression', 'train_secs': 177.1, 'AUC': 0.9456, 'F1': 0.8639, 'Accuracy': 0.8772, 'best_params': {'elasticNetParam': 0.0, 'maxIter': 50, 'regParam': 0.1}}


## 3.2 Naive Bayes

In [6]:
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import MinMaxScaler
from pyspark.sql.functions import udf, col
from pyspark.ml.linalg import Vectors, VectorUDT, SparseVector, DenseVector # Import SparseVector and DenseVector
from pyspark.ml.param.shared import HasInputCol, HasOutputCol
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable, Identifiable

# Define a UDF to clip negative values in a vector to 0
# It takes a Spark ML VectorUDT (either DenseVector or SparseVector) and returns a VectorUDT
@udf(VectorUDT())
def clip_vector_udf(vector):
    if vector is None:
        return None
    # Use the directly imported SparseVector and DenseVector for type checking
    if isinstance(vector, SparseVector):
        # For sparse vectors, only modify non-zero values (indices correspond to values)
        new_values = [max(0.0, v) for v in vector.values]
        # Reconstruct sparse vector with clipped values
        return Vectors.sparse(vector.size, vector.indices, new_values)
    elif isinstance(vector, DenseVector):
        # For dense vectors, clip all values
        new_values = [max(0.0, v) for v in vector.values]
        return Vectors.dense(new_values)
    else:
        # Should not reach here for standard Spark ML vectors
        return vector

# Create a custom transformer to apply the UDF
class NonNegativeFeatureClipper(Transformer, HasInputCol, HasOutputCol,
                                DefaultParamsReadable, DefaultParamsWritable):

    def __init__(self, inputCol="features", outputCol="clippedFeatures", uid=None):
        # Call Transformer's __init__ via super() without passing uid explicitly.
        # Transformer's __init__ will handle uid generation if uid is None.
        # This prevents uid from being passed to mixins like HasInputCol which don't expect it.
        super(NonNegativeFeatureClipper, self).__init__()

        # Set the uid after the super().__init__() call, if a specific uid was provided.
        # If uid was None, Transformer's __init__ would have already set a random one.
        if uid is not None:
            self.uid = uid

        # Set default values for the parameters provided by mixins (HasInputCol, HasOutputCol)
        self._setDefault(inputCol="features", outputCol="clippedFeatures")

        # Set the actual values for the parameters
        self._set(inputCol=inputCol, outputCol=outputCol)

    def _transform(self, dataset):
        input_col = self.getInputCol()
        output_col = self.getOutputCol()
        return dataset.withColumn(output_col, clip_vector_udf(col(input_col)))

    def _copy(self, extra=None):
        return self.copyValues(NonNegativeFeatureClipper(inputCol=self.getInputCol(),
                                                        outputCol=self.getOutputCol(),
                                                        uid=self.uid), extra)

# Instantiate the custom clipper
clipper = NonNegativeFeatureClipper(inputCol="features", outputCol="clippedFeatures")

# Create a MinMaxScaler to ensure non-negative features for Naive Bayes (it will scale the clipped features)
# This scales features to [0, 1] range.
saler = MinMaxScaler(inputCol="clippedFeatures", outputCol="scaledFeatures")

# Initialize Naive Bayes with the scaled features
nb_estimator = NaiveBayes(featuresCol="scaledFeatures", labelCol="label", modelType="multinomial")

# Build a Pipeline: clipper -> saler -> NaiveBayes
nb_pipeline = Pipeline(stages=[clipper, saler, nb_estimator])

# Adjust the ParamGridBuilder to target the NaiveBayes stage within the pipeline
nb_grid = ParamGridBuilder().addGrid(nb_estimator.smoothing,[0.5,1.0]).build()

# Tune the pipeline instead of just the NaiveBayes estimator
nb_best,nb_row,nb_pred=tune("NaiveBayes",nb_pipeline,nb_grid, train)

NaiveBayes {'model': 'NaiveBayes', 'train_secs': 545.2, 'AUC': 0.6017, 'F1': 0.8178, 'Accuracy': 0.8364, 'best_params': {}}


## 3.3 Linear SVM

In [7]:
svc=LinearSVC(featuresCol="features", labelCol="label", maxIter=50)
svc_grid=ParamGridBuilder().addGrid(svc.regParam,[0.01,0.1]).build()
svc_best,svc_row,svc_pred=tune("LinearSVC",svc,svc_grid, train)

LinearSVC {'model': 'LinearSVC', 'train_secs': 136.3, 'AUC': 0.9389, 'F1': 0.886, 'Accuracy': 0.8922, 'best_params': {'maxIter': 50, 'regParam': 0.1}}


## 3.4 Random Forest

In [8]:
import time
from pyspark.ml.feature import ChiSqSelector, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import udf, col
from pyspark.ml.linalg import Vectors, VectorUDT, SparseVector, DenseVector

auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

# Define a UDF to strip metadata from a vector by reconstructing it
# This ensures that any problematic metadata marking features as categorical is removed.
@udf(VectorUDT())
def strip_vector_metadata_udf(vec):
    if vec is None:
        return None
    if isinstance(vec, SparseVector):
        return Vectors.sparse(vec.size, vec.indices, vec.values)
    elif isinstance(vec, DenseVector):
        return Vectors.dense(vec.values)
    # Fallback for unexpected types, though VectorUDT generally ensures it's one of the above.
    return vec

# 20k sparse dims -> 1000 most informative, on a 10% sample (RF can't handle full sparse text)
rf_train = train.sample(False, 0.10, seed=42)
sel = ChiSqSelector(numTopFeatures=500, featuresCol="features",
                    labelCol="label", outputCol="selectedFeatures") # Renamed outputCol for clarity
sel_model = sel.fit(rf_train)

# Use VectorAssembler to create a new feature column
assembler = VectorAssembler(inputCols=["selectedFeatures"], outputCol="assembledFeatures")

rf_tr   = sel_model.transform(rf_train)
rf_tr   = assembler.transform(rf_tr)
# Explicitly strip any lingering metadata from assembledFeatures to prevent misinterpretation as categorical
rf_tr = rf_tr.withColumn("assembledFeatures", strip_vector_metadata_udf(col("assembledFeatures"))).cache()

rf_test = sel_model.transform(test)
rf_test = assembler.transform(rf_test)
# Explicitly strip any lingering metadata for test data as well
rf_test = rf_test.withColumn("assembledFeatures", strip_vector_metadata_udf(col("assembledFeatures")))

# Removed VectorIndexer. RandomForestClassifier handles Vector type features from VectorAssembler as continuous by default,
# now that metadata is explicitly cleaned.

rf = RandomForestClassifier(featuresCol="assembledFeatures", labelCol="label", # Use assembledFeatures directly
                            numTrees=30, maxDepth=8, maxBins=32,
                            maxMemoryInMB=512, seed=42)
t0=time.time(); rf_model=rf.fit(rf_tr); secs=round(time.time()-t0,1)
rf_pred=rf_model.transform(rf_test)

# Manually create rf_row to match the format of other models for the summary table
rf_row = {
    "model": "RandomForest",
    "train_secs": secs,
    "AUC": round(auc_eval.evaluate(rf_pred), 4),
    "F1": None, # F1 and Accuracy are not computed by this cell's evaluation
    "Accuracy": None, # Fill with actual values if calculated later
    "best_params": {
        "numTrees": rf_model.numTrees,
        "maxDepth": rf_model.maxDepth
        # Add other relevant parameters if desired
    }
}

print("RandomForest AUC:", rf_row['AUC'], "| train_secs:", rf_row['train_secs'])

RandomForest AUC: 0.8609 | train_secs: 13.4


In [9]:
rf=RandomForestClassifier(featuresCol="features", labelCol="label")
# Further reduced numTrees and maxDepth, and simplified subsamplingRate and featureSubsetStrategy to decrease complexity and resource usage
rf_grid=(ParamGridBuilder().addGrid(rf.numTrees,[10]) # Reduced to a single, smaller value
         .addGrid(rf.maxDepth,[2]) # Reduced to a single, smaller value
         .addGrid(rf.subsamplingRate, [0.7]) # Simplified to a single value
         .addGrid(rf.featureSubsetStrategy, ['sqrt']) # Simplified to a single value
         .build())
rf_best,rf_row,rf_pred=tune("RandomForest",rf,rf_grid, train)

RandomForest {'model': 'RandomForest', 'train_secs': 212.7, 'AUC': 0.5625, 'F1': 0.6915, 'Accuracy': 0.7857, 'best_params': {'maxDepth': 2, 'numTrees': 10}}


## 3.5 Summary table (screenshot) + save models

In [10]:
import pandas as pd
summary=pd.DataFrame([lr_row,nb_row,svc_row,rf_row])[
    ["model","best_params","train_secs","AUC","F1","Accuracy"]]
summary.to_csv(EXPORT+"/model_summary.csv", index=False); display(summary)
for n,m in [("lr",lr_best),("nb",nb_best),("svc",svc_best),("rf",rf_best)]:
    m.write().overwrite().save(BASE+f"/models/{n}")
print("models saved")

,model,best_params,train_secs,AUC,F1,Accuracy
0,LogisticRegression,"{'elasticNetParam': 0.0, 'maxIter': 50, 'regPa...",177.1,0.9456,0.8639,0.8772
1,NaiveBayes,{},545.2,0.6017,0.8178,0.8364
2,LinearSVC,"{'maxIter': 50, 'regParam': 0.1}",136.3,0.9389,0.8860,0.8922
3,RandomForest,"{'maxDepth': 2, 'numTrees': 10}",212.7,0.5625,0.6915,0.7857


models saved


## Version control
**>=3 commits per model** (baseline, ParamGrid, tuned metrics). Branch-per-model rewarded: `model/logreg`, `model/naive-bayes`, ...